Udemy Purchase Scraper

In [ ]:
# Change this to match your version of Chrome if the automatically downloaded version is too new
chrome_version = None #153

# I worry that all of these xpath links will get broken by the react they're using. They had strange names.

# Change this variable if you're running all and want to overwrite existing cookies
force_new_cookies = False

# Parameters
cookies_path = "cookies_2.json"

# Options for selenium
import undetected_chromedriver as uc
from undetected_chromedriver import By
from typing import Union, Tuple, Any
import pandas as pd
import time
import json
import os
import datetime
from decimal import Decimal
import re
import random

options = uc.ChromeOptions()

if chrome_version is not None:
    driver = uc.Chrome(use_subprocess=False, options=options, version_main=chrome_version)
else:
    driver = uc.Chrome(use_subprocess=False, options=options)

# Go to udemy
driver.get("https://www.udemy.com/")

Manually log into Udemy now because trying to log in automatically with 2FA and everything will be a pain in the ass. Save the cookies to skip this step later (and prevent being locked out if you log in too many times).

In [ ]:
def saveCookies(driver: uc.Chrome, f_path: str, force_cookies: bool = True) -> None:
    '''
    Saves the cookies from your udemy in

    :param driver: The web driver
    :type driver: Chrome

    :param f_path: The file name (or full file path)
    :type f_path: str

    :param force_cookies: If we force the cookies even if they already exist, defaults to [True]
    :type force_cookies: bool

    :return: Nothing
    :rtype: None
    '''
    # Checking if the cookies file already exists
    if(force_cookies or not os.path.exists(f_path)):
        # Wait for an input for the user to log in
        input("Continue after you've logged into Udemy!")
        # To save cookies
        saved_cookies = driver.get_cookies()
        with open(f_path, "w") as file:
            json.dump(saved_cookies, file)
    return None

In [ ]:
def loadCookies(driver: uc.Chrome, f_path: str):
    '''
    Loads your Udemy cookies in

    :param driver: The web driver
    :type driver: Chrome

    :param f_path: The file name (or full file path)
    :type f_path: str

    :return: Nothing
    :rtype: None
    '''
    # To load in cookies
    with open(f_path, "r") as file:
        saved_cookies = json.load(file)
    for x in saved_cookies:
        driver.add_cookie(x)
    # Wait for one second just to be sure (probably not needed)
    time.sleep(1)
    return None

In [ ]:
saveCookies(driver=driver, f_path=cookies_path, force_cookies=force_new_cookies)
loadCookies(driver=driver, f_path=cookies_path)

In [ ]:
driver.get("https://www.udemy.com/dashboard/purchase-history/")

In [ ]:
from dataclasses import dataclass
import datetime

In [ ]:
@dataclass
class UdemyCourse:
    '''
    A class to store Udemy courses
    '''
    course_name: str
    course_link: str
    course_cost: float
    course_author: str|None = None

In [ ]:
@dataclass
class UdemyPurchase:
    '''
    A class to store udemy purchases
    '''
    purchase_number: str
    purchase_date: datetime.date
    purchase_cost: float

In [ ]:
def getPurchasePage(driver: any) -> any:
    '''
    A function to get the purchases from the 'purchase-history' page
    '''
    purchase_table = driver.find_element(By.XPATH, "//table[@class='paginated-table_purchase-history-table__ncCll']")
    purchases = purchase_table.find_elements(By.XPATH, ".//tr[@class='paginated-table_purchase-history-table--main-row__5hHCb']")
    # for ind, purchase in enumerate(purchases):
    #     purchase[ind] = purchase.find_element(By.XPATH, ".//div[@class='paginated-table_purchase-details--text-container__EvFRz']")
    return purchases

In [ ]:
def getPurchaseInformation(purchase: any) -> any:
    '''
    
    '''
    purchase_number = purchase.find_element(By.XPATH, ".//span[@class='ud-btn-label' and contains(., 'Receipt')]")
    purchase_number = purchase_number.find_element(By.XPATH, "..")
    purchase_number = purchase_number.get_attribute("href")
    purchase_number = purchase_number.split("/")[-2]

    purchase_cost = purchase.find_element(By.XPATH, ".//div[@class='paginated-table_purchase-history-table--data-cell__LO_7L']/div[@class='app_positive-text___Wb07']").text
    purchase_cost = Decimal(purchase_cost.replace("$","").strip())

    purchase_date = purchase.find_element(By.XPATH, ".//div[@class='paginated-table_purchase-history-table--data-cell__LO_7L' and not(descendant::*)]").text
    purchase_date = datetime.datetime.strptime(purchase_date, "%b %d, %Y").date()


    cur_purchase = UdemyPurchase(purchase_number=purchase_number,
                                 purchase_cost= purchase_cost,
                                 purchase_date=purchase_date)
    return(cur_purchase)

In [ ]:
class MultiplePurchaseError(Exception):
    '''Raised when a purchase only has a single course'''

def checkMultipleCourses(purchase: any) -> any:
    '''
    A function to check if a purchase has multiple courses
    '''
    try:
        link_section = purchase.find_element(By.XPATH, ".//div[@class='paginated-table_purchase-details--text-container__EvFRz']")
        show_more_button = link_section.find_element(By.XPATH, ".//button[@class='ud-btn ud-btn-medium ud-btn-link ud-btn-text-sm show-more_focusable-label__dmEhW']")
        return(show_more_button)
    except:
        raise MultiplePurchaseError("Purchase only has a single course")

In [ ]:
def getPurchaseCourseSingle(purchase: any) -> any:
    '''
    A function to get the courses from a purchase with a single course
    '''
    course_name_element = purchase.find_element(By.XPATH, ".//div[@class='paginated-table_purchase-details--text__JKVm6 paginated-table_purchase-details--content-title-text__DxDuf ud-text-md']")
    course_name_element = course_name_element.find_element(By.XPATH, ".//a")

    course_name = course_name_element.get_attribute("innerHTML")
    course_link = course_name_element.get_attribute("href")
    
    course_price = purchase.find_element(By.XPATH, ".//div[@class='app_positive-text___Wb07']").text
    course_price = Decimal(course_price.replace("$","").strip())
    
    course = UdemyCourse(course_name=course_name,
                        course_link=course_link,
                        course_cost=course_price,
                        course_author="Fix Later")
    return(course)


In [ ]:
def getPurchaseCourseMultiple(driver: any) -> any:
    '''
    A function to get the courses from a purchase with multiple courses
    '''
    courses = driver.find_elements(By.XPATH, ".//tr[@class='paginated-table_purchase-history-table--sub-row__2g87b']")
    courses = [getPurchaseCourseSingle(purchase=course) for course in courses]
    return(courses)
    

In [ ]:
def getPurchaseCourses(purchase: any, driver: any) -> any:
    '''
    A function to get the courses in a purchase
    '''
    try:
        expand_purchase = checkMultipleCourses(purchase)
        expand_purchase.click()
        courses = getPurchaseCourseMultiple(driver=driver)
        expand_purchase.click()
    except MultiplePurchaseError:
        courses = [getPurchaseCourseSingle(purchase)]
    return(courses)
        

In [ ]:
purchase_page = getPurchasePage(driver=driver)
for purchase in purchase_page:
    print(getPurchaseInformation(purchase=purchase))
    print(getPurchaseCourses(purchase=purchase, driver=driver))

In [ ]:
getPurchaseCourses(purchase=purchase_page[0], driver=driver)